# SOMAX: Training & Evaluation

**Step 2 of 2 — Benchmarking, training, and GGUF export**

**Prerequisite:** Run `scripts/download.py` first to download the WAXAL dataset.

**Pipeline:**
1. Setup & Dependencies
2. Baseline Token Fertility
3. Train Router Classifier
4. Train Unified BPE Vocabulary
5. Train LoRA Variant (A–E)
6. Benchmark Fertility Reduction
7. Export to GGUF

## 1. Setup & Dependencies

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
# Check GPU
!nvidia-smi

In [ ]:
# Skip torch and psutil (already in Colab)
# Focus on the research-specific extensions
!uv pip install -U transformers datasets accelerate peft bitsandbytes llama-cpp-python

In [ ]:
import os
from pathlib import Path

if not os.path.exists('somax'):
    !git clone 'https://github.com/attabeezy/somax.git'
    %cd somax
else:
    %cd somax

In [ ]:
import os

# ── Configuration ─────────────────────────────────────────
LANGUAGE    = "akan"                      # proof-of-concept language
TRAIN_GROUP = "D"                         # primary hypothesis: TTS → ASR → TTS
BASE_MODEL  = "meta-llama/Llama-3.2-1B"
DATA_DIR    = f"data/{LANGUAGE}"
CONFIG_FILE = "configs/variants.yaml"
# ──────────────────────────────────────────────────────────

HF_TOKEN = os.environ.get("HF_TOKEN", "SIKE TOKEN NOT SET")
if not HF_TOKEN:
    print("WARNING: HF_TOKEN not set. Required for Llama models.")
    print("Set it with:  os.environ['HF_TOKEN'] = 'your_token'")

In [ ]:
import shutil

def setup_drive() -> str | None:
    """Mount Google Drive and return results base path."""
    try:
        from google.colab import drive
        drive.mount("/content/drive", timeout_ms=120000)
        base = "/content/drive/MyDrive/WAXAL-results"
        os.makedirs(base, exist_ok=True)
        print(f"Google Drive mounted: {base}")
        return base
    except Exception as e:
        print(f"Google Drive not available: {e}")
        return None

def save_to_drive(source_dir: str, drive_base: str | None, label: str) -> None:
    """Copy a local directory to Google Drive."""
    if not drive_base or not os.path.exists(source_dir):
        return
    dest = os.path.join(drive_base, label)
    if os.path.exists(dest):
        shutil.rmtree(dest)
    shutil.copytree(source_dir, dest)
    print(f"Saved to Drive: {dest}")

DRIVE_BASE = setup_drive()

# Download dataset if not present
if not Path(DATA_DIR).exists() or not list(Path(DATA_DIR).glob("*.jsonl")):
    print("Dataset not found. Downloading WAXAL dataset...")
    !python scripts/download.py --lang {LANGUAGE} --output data/
else:
    print(f"Dataset found: {DATA_DIR}")

## 2. Baseline Token Fertility

In [ ]:
!python scripts/benchmark_fertility.py \
    --tokenizer {BASE_MODEL} \
    --test-file {DATA_DIR}/twi_tts_test.jsonl \
    --output results/baseline_fertility.json

## 3. Train Router Classifier

In [ ]:
!python scripts/train_router.py \
    --data {DATA_DIR} \
    --output models/router/ \
    --language {LANGUAGE}

## 4. Train Unified BPE Vocabulary

In [ ]:
!python scripts/train_bpe.py \
    --input {DATA_DIR} \
    --output models/tokenizers/ \
    --language {LANGUAGE} \
    --vocab-size 8000

## 5. Train LoRA Variant

Edit `TRAIN_GROUP` in the config cell above to run a different variant:

| Group | Sequence | Notes |
|-------|----------|-------|
| A | ASR only | |
| B | TTS only | |
| C | Mixed | |
| **D** | TTS → ASR → TTS | Primary hypothesis |
| E | ASR → TTS | |

In [ ]:
!python scripts/train_lora.py \
    --group {TRAIN_GROUP} \
    --model {BASE_MODEL} \
    --data {DATA_DIR} \
    --output checkpoints/ \
    --config {CONFIG_FILE}

In [ ]:
save_to_drive("checkpoints", DRIVE_BASE, "checkpoints")

## 6. Benchmark Fertility Reduction

In [ ]:
!python scripts/benchmark_fertility.py \
    --tokenizer {BASE_MODEL} \
    --waxal-tokenizer models/tokenizers/{LANGUAGE}/unified_tokenizer.json \
    --test-file {DATA_DIR}/twi_tts_test.jsonl \
    --compare \
    --output results/fertility_comparison.json

## 7. Export to GGUF (for Edge Deployment)

In [ ]:
!python scripts/export_gguf.py \
    --checkpoint checkpoints/variant_{TRAIN_GROUP}/final/ \
    --output models/gguf/ \
    --quantization Q4_K_M

In [ ]:
save_to_drive("results", DRIVE_BASE, "results")
save_to_drive("models", DRIVE_BASE, "models")

## Summary

In [ ]:
import json
from pathlib import Path

print("=" * 50)
print("WAXAL-Dual-Core Pipeline Complete!")
print("=" * 50)
print(f"Language:       {LANGUAGE}")
print(f"Variant:        {TRAIN_GROUP}")

results_path = Path("results/fertility_comparison.json")
if results_path.exists():
    r = json.loads(results_path.read_text())
    print(f"\nBaseline F:     {r['baseline']['fertility_mean']:.3f} tokens/word")
    print(f"WAXAL F:        {r['waxal']['fertility_mean']:.3f} tokens/word")
    print(f"Reduction:      {r['reduction_pct']:.1f}%  (target: ≥30%)")
else:
    baseline_path = Path("results/baseline_fertility.json")
    if baseline_path.exists():
        r = json.loads(baseline_path.read_text())
        print(f"\nBaseline F:     {r['fertility_mean']:.3f} tokens/word")

print("\nNext steps:")
print("  1. Download results from Google Drive")
print("  2. Run scripts/benchmark_inference.py on Dell Latitude 7400")
print("  3. Run scripts/benchmark_fertility.py --compare on edge hardware")